# ETL Pipeline — Alquiler Barcelona

Pipeline completo de extracción, transformación y carga de datos de alquiler en Barcelona.

| Etapa | Fuente | Salida |
|-------|--------|--------|
| **EXTRACT** | Excel Incasol (`extract_incasol/`) | `district_panel_df`, `rental_panel_df` |
| **TRANSFORM** | IPC INE, Euribor BDE, IST Idescat, Open Data BCN (`extract_ipc_ist_merge/`) | `districts_final`, `neighborhood_final` |
| **LOAD** | DataFrames en memoria | `df_distritos_eda`, `df_barrios_eda` |

In [1]:
%pip install openpyxl pyarrow statsmodels scipy requests --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#  Imports 
import os
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from openpyxl import load_workbook
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pyarrow.parquet as pq

plt.style.use("seaborn-v0_8")

# Path configuration 

EXTRACT_DIR = "extract_incasol"
TRANSFORM_DIR = "extract_ipc_ist_merge"

CONTRACTS_FILE  = os.path.join(EXTRACT_DIR, "trimestral_bcn_contractes.xlsx")
RENT_FILE       = os.path.join(EXTRACT_DIR, "trimestral_bcn_lloguer.xlsx")
RENT_M2_FILE    = os.path.join(EXTRACT_DIR, "trimestral_bcn_lloguer_m2.xlsx")
SURFACE_FILE    = os.path.join(EXTRACT_DIR, "trimestral_bcn_sup.xlsx")

IPC_FILE     = os.path.join(TRANSFORM_DIR, "ipc_ine.csv")
EURIBOR_FILE = os.path.join(TRANSFORM_DIR, "SeriesBIE[21].csv")
IST_FILE     = os.path.join(TRANSFORM_DIR, "ist14058mun.csv")

# EXTRACT
Lectura de los cuatro Excel de Incasol (contratos, alquiler medio, alquiler €/m², superficie media) a nivel de **barrio** y **distrito**.

## Helper functions — Barrios y Districtes

In [3]:
# Shared utilities 

def clean_text(value):
    if value is None:
        return ""
    return " ".join(str(value).strip().split())


def parse_nullable_number(value, number_type="float"):
    if value is None:
        return None

    number_type = str(number_type).strip().lower()
    if number_type in {"int", "integer"}:
        number_type = "int"
    elif number_type in {"float", "double", "decimal"}:
        number_type = "float"
    else:
        raise ValueError("number_type debe ser 'int' o 'float'")

    if isinstance(value, (int, float)):
        return int(value) if number_type == "int" else float(value)

    text = str(value).strip()
    if text == "":
        return None
    if text.lower() in {"n.d.", "nd", "n.d", "na", "n/a", "-"}:
        return None

    text = text.replace(",", ".")
    try:
        return int(float(text)) if number_type == "int" else float(text)
    except ValueError:
        return None


def get_year_sheets(filepath, start_year=2000, end_year=2025):
    wb = load_workbook(filepath, data_only=True)
    valid_sheets = [
        s for s in wb.sheetnames
        if str(s).isdigit() and start_year <= int(s) <= end_year
    ]
    return sorted(valid_sheets, key=int)


#  Barrio extraction 

def extract_barris_one_sheet_df(filepath, sheet_name, value_col_name, number_type="float"):
    wb = load_workbook(filepath, data_only=True)
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))

    barris_row_idx = None
    for i, row in enumerate(rows):
        second_cell = clean_text(row[1]) if len(row) > 1 else ""
        if second_cell.startswith("Barris"):
            barris_row_idx = i
            break

    if barris_row_idx is None:
        raise ValueError(f"No encontré la fila 'Barris' en la hoja {sheet_name}.")

    year = int(sheet_name)
    quarter_cols = {2: 1, 3: 2, 4: 3, 5: 4}
    result = []

    for i in range(barris_row_idx + 1, len(rows)):
        row = rows[i]
        code = row[0] if len(row) > 0 else None
        neighborhood = clean_text(row[1]) if len(row) > 1 else ""

        if code is None or neighborhood == "" or not isinstance(code, int):
            break

        for col_idx, quarter in quarter_cols.items():
            value = row[col_idx] if col_idx < len(row) else None
            result.append({
                "neighborhood_code": code,
                "neighborhood": neighborhood,
                "year": year,
                "quarter": quarter,
                value_col_name: parse_nullable_number(value, number_type=number_type),
            })

    df = pd.DataFrame(result)
    return df.sort_values(["year", "neighborhood_code", "quarter"]).reset_index(drop=True)


def extract_barris_all_years_df(filepath, value_col_name, number_type="float",
                                 start_year=2014, end_year=2025):
    dfs = [
        extract_barris_one_sheet_df(filepath, s, value_col_name, number_type)
        for s in get_year_sheets(filepath, start_year, end_year)
    ]
    df = pd.concat(dfs, ignore_index=True)
    return df.sort_values(["year", "neighborhood_code", "quarter"]).reset_index(drop=True)


def validate_panel_df(df, value_col_name):
    print(f"== Validación: {value_col_name} ==")
    dups = df.duplicated(subset=["neighborhood_code", "year", "quarter"], keep=False).sum()
    print(f"  Duplicados: {int(dups)}")
    total_missing = int(df[value_col_name].isna().sum())
    print(f"  Missing {value_col_name}: {total_missing} de {len(df)}")
    print(f"  Barrios únicos: {df['neighborhood_code'].nunique()} | "
          f"Años: {df['year'].min()}–{df['year'].max()}\n")


#  District extraction 

def parse_old_district_label(text):
    text = clean_text(text)
    match = re.match(r"^(\d+)\.\s*(.+)$", text)
    if not match:
        return None, None
    return int(match.group(1)), match.group(2).strip()


def extract_districts_new_format(rows, year, value_col_name, number_type="float"):
    district_start_idx = None
    for i, row in enumerate(rows):
        second_cell = clean_text(row[1]) if len(row) > 1 else ""
        if second_cell.startswith("Districtes municipals"):
            district_start_idx = i
            break

    if district_start_idx is None:
        raise ValueError(f"No encontré 'Districtes municipals' para {year}.")

    quarter_cols = {2: 1, 3: 2, 4: 3, 5: 4}
    result = []

    for i in range(district_start_idx + 1, len(rows)):
        row = rows[i]
        code = row[0] if len(row) > 0 else None
        district = clean_text(row[1]) if len(row) > 1 else ""

        if code is None or district == "" or not isinstance(code, int):
            break

        for col_idx, quarter in quarter_cols.items():
            value = row[col_idx] if col_idx < len(row) else None
            result.append({
                "district_code": code,
                "district": district,
                "year": year,
                "quarter": quarter,
                value_col_name: parse_nullable_number(value, number_type=number_type),
            })

    return result


def extract_districts_old_format(rows, year, value_col_name, number_type="float"):
    quarter_cols = {1: 1, 2: 2, 3: 3, 4: 4}
    result = []

    for row in rows:
        first_cell = clean_text(row[0]) if len(row) > 0 else ""
        if first_cell == "" or first_cell == "Barcelona":
            continue

        district_code, district = parse_old_district_label(first_cell)
        if district_code is None:
            continue

        for col_idx, quarter in quarter_cols.items():
            value = row[col_idx] if col_idx < len(row) else None
            result.append({
                "district_code": district_code,
                "district": district,
                "year": year,
                "quarter": quarter,
                value_col_name: parse_nullable_number(value, number_type=number_type),
            })

    return result


def extract_districts_one_sheet_df(filepath, sheet_name, value_col_name, number_type="float"):
    wb = load_workbook(filepath, data_only=True)
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    year = int(sheet_name)

    result = (
        extract_districts_new_format(rows, year, value_col_name, number_type)
        if year >= 2014
        else extract_districts_old_format(rows, year, value_col_name, number_type)
    )

    df = pd.DataFrame(result)
    return df.sort_values(["year", "district_code", "quarter"]).reset_index(drop=True)


def extract_districts_all_years_df(filepath, value_col_name, number_type="float",
                                    start_year=2000, end_year=2025):
    dfs = [
        extract_districts_one_sheet_df(filepath, s, value_col_name, number_type)
        for s in get_year_sheets(filepath, start_year, end_year)
    ]
    df = pd.concat(dfs, ignore_index=True)
    return df.sort_values(["year", "district_code", "quarter"]).reset_index(drop=True)


def validate_district_panel_df(df, value_col_name):
    print(f"== Validación distritos: {value_col_name} ==")
    dups = df.duplicated(subset=["district_code", "year", "quarter"], keep=False).sum()
    print(f"  Duplicados: {int(dups)}")
    total_missing = int(df[value_col_name].isna().sum())
    print(f"  Missing {value_col_name}: {total_missing} de {len(df)}")
    print(f"  Distritos únicos: {df['district_code'].nunique()} | "
          f"Años: {df['year'].min()}–{df['year'].max()}\n")

## Extracción — Barrios (2014–2025)

In [4]:
contracts_barris_df = extract_barris_all_years_df(
    CONTRACTS_FILE, "num_contracts", number_type="int", start_year=2014, end_year=2025
)

avg_rent_barris_df = extract_barris_all_years_df(
    RENT_FILE, "avg_rent", number_type="float", start_year=2014, end_year=2025
)

avg_rent_m2_barris_df = extract_barris_all_years_df(
    RENT_M2_FILE, "avg_rent_m2", number_type="float", start_year=2014, end_year=2025
)

avg_surface_barris_df = extract_barris_all_years_df(
    SURFACE_FILE, "avg_surface", number_type="float", start_year=2014, end_year=2025
)

contracts_barris_df["num_contracts"] = contracts_barris_df["num_contracts"].astype("Int64")

for df, col in [
    (contracts_barris_df,   "num_contracts"),
    (avg_rent_barris_df,    "avg_rent"),
    (avg_rent_m2_barris_df, "avg_rent_m2"),
    (avg_surface_barris_df, "avg_surface"),
]:
    validate_panel_df(df, col)

== Validación: num_contracts ==
  Duplicados: 0
  Missing num_contracts: 75 de 3504
  Barrios únicos: 73 | Años: 2014–2025

== Validación: avg_rent ==
  Duplicados: 0
  Missing avg_rent: 221 de 3504
  Barrios únicos: 73 | Años: 2014–2025

== Validación: avg_rent_m2 ==
  Duplicados: 0
  Missing avg_rent_m2: 219 de 3504
  Barrios únicos: 73 | Años: 2014–2025

== Validación: avg_surface ==
  Duplicados: 0
  Missing avg_surface: 104 de 3504
  Barrios únicos: 73 | Años: 2014–2025



## Merge — Panel barrios

In [5]:
KEY_BARRIS = ["neighborhood_code", "year", "quarter"]

rental_panel_df = (
    contracts_barris_df
    .merge(avg_rent_barris_df[KEY_BARRIS + ["avg_rent"]],
           on=KEY_BARRIS, how="left", validate="one_to_one")
    .merge(avg_rent_m2_barris_df[KEY_BARRIS + ["avg_rent_m2"]],
           on=KEY_BARRIS, how="left", validate="one_to_one")
    .merge(avg_surface_barris_df[KEY_BARRIS + ["avg_surface"]],
           on=KEY_BARRIS, how="left", validate="one_to_one")
)

rental_panel_df = rental_panel_df[
    ["neighborhood_code", "neighborhood", "year", "quarter",
     "num_contracts", "avg_rent", "avg_rent_m2", "avg_surface"]
].sort_values(["year", "neighborhood_code", "quarter"]).reset_index(drop=True)

print(rental_panel_df.info())
rental_panel_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   neighborhood_code  3504 non-null   int64  
 1   neighborhood       3504 non-null   object 
 2   year               3504 non-null   int64  
 3   quarter            3504 non-null   int64  
 4   num_contracts      3429 non-null   Int64  
 5   avg_rent           3283 non-null   float64
 6   avg_rent_m2        3285 non-null   float64
 7   avg_surface        3400 non-null   float64
dtypes: Int64(1), float64(3), int64(3), object(1)
memory usage: 222.6+ KB
None


,neighborhood_code,neighborhood,year,quarter,num_contracts,avg_rent,avg_rent_m2,avg_surface
0,1,el Raval,2014,1,356,589.548624,10.76,60.44663
1,1,el Raval,2014,2,409,550.631711,10.52,57.37912
2,1,el Raval,2014,3,423,576.454232,9.84,65.05985
3,1,el Raval,2014,4,395,596.995494,10.81,59.96962
4,2,el Barri Gòtic,2014,1,135,712.786519,10.58,78.13333


## Extracción — Distritos (2000–2025)

In [6]:
contracts_district_df = extract_districts_all_years_df(
    CONTRACTS_FILE, "num_contracts", number_type="int", start_year=2000, end_year=2025
)

avg_rent_district_df = extract_districts_all_years_df(
    RENT_FILE, "avg_rent", number_type="float", start_year=2000, end_year=2025
)

avg_rent_m2_district_df = extract_districts_all_years_df(
    RENT_M2_FILE, "avg_rent_m2", number_type="float", start_year=2000, end_year=2025
)

avg_surface_district_df = extract_districts_all_years_df(
    SURFACE_FILE, "avg_surface", number_type="float", start_year=2000, end_year=2025
)

contracts_district_df["num_contracts"] = contracts_district_df["num_contracts"].astype("Int64")

for df, col in [
    (contracts_district_df,   "num_contracts"),
    (avg_rent_district_df,    "avg_rent"),
    (avg_rent_m2_district_df, "avg_rent_m2"),
    (avg_surface_district_df, "avg_surface"),
]:
    validate_district_panel_df(df, col)

== Validación distritos: num_contracts ==
  Duplicados: 0
  Missing num_contracts: 10 de 1040
  Distritos únicos: 10 | Años: 2000–2025

== Validación distritos: avg_rent ==
  Duplicados: 0
  Missing avg_rent: 10 de 1040
  Distritos únicos: 10 | Años: 2000–2025

== Validación distritos: avg_rent_m2 ==
  Duplicados: 0
  Missing avg_rent_m2: 10 de 1040
  Distritos únicos: 10 | Años: 2000–2025

== Validación distritos: avg_surface ==
  Duplicados: 0
  Missing avg_surface: 10 de 1040
  Distritos únicos: 10 | Años: 2000–2025



## Merge — Panel distritos

In [7]:
KEY_DIST = ["district_code", "year", "quarter"]

district_panel_df = (
    contracts_district_df
    .merge(avg_rent_district_df[KEY_DIST + ["avg_rent"]],
           on=KEY_DIST, how="left", validate="one_to_one")
    .merge(avg_rent_m2_district_df[KEY_DIST + ["avg_rent_m2"]],
           on=KEY_DIST, how="left", validate="one_to_one")
    .merge(avg_surface_district_df[KEY_DIST + ["avg_surface"]],
           on=KEY_DIST, how="left", validate="one_to_one")
)

district_panel_df = district_panel_df[
    ["district_code", "district", "year", "quarter",
     "num_contracts", "avg_rent", "avg_rent_m2", "avg_surface"]
].sort_values(["year", "district_code", "quarter"]).reset_index(drop=True)

print(district_panel_df.info())
district_panel_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   district_code  1040 non-null   int64  
 1   district       1040 non-null   object 
 2   year           1040 non-null   int64  
 3   quarter        1040 non-null   int64  
 4   num_contracts  1030 non-null   Int64  
 5   avg_rent       1030 non-null   float64
 6   avg_rent_m2    1030 non-null   float64
 7   avg_surface    1030 non-null   float64
dtypes: Int64(1), float64(3), int64(3), object(1)
memory usage: 66.1+ KB
None


,district_code,district,year,quarter,num_contracts,avg_rent,avg_rent_m2,avg_surface
0,1,Ciutat Vella,2000,1,397,286.9246,4.998756,60.976331
1,1,Ciutat Vella,2000,2,355,301.3140,5.401631,65.389937
2,1,Ciutat Vella,2000,3,373,338.7222,5.958517,66.824390
3,1,Ciutat Vella,2000,4,415,370.0641,5.758672,65.384211
4,2,Eixample,2000,1,1200,425.8277,5.486564,79.968954


## Checkpoint — guardar parquets intermedios

In [8]:
# rental_panel_df.to_parquet("rental_panel_barris_2014_2025.parquet", index=False)
# rental_panel_df.to_csv("rental_panel_barris_2014_2025.csv", index=False, encoding="utf-8-sig")

# district_panel_df.to_parquet("district_panel_2000_2025.parquet", index=False)
# district_panel_df.to_csv("district_panel_2000_2025.csv", index=False, encoding="utf-8-sig")

print("Checkpoint EXTRACT en memoria (guardado en disco desactivado).")

Checkpoint EXTRACT en memoria (guardado en disco desactivado).


# TRANSFORM
Enriquecimiento de los paneles con IPC (INE), Euríbor 12M (BDE), IST (Idescat) y densidad de población (Open Data BCN).

## Variable temporal y features base

In [9]:
# Los paneles de EXTRACT ya están en memoria; los usamos directamente.
districts_raw    = district_panel_df.copy()
neighborhood_raw = rental_panel_df.copy()


def create_time_index(df):
    df = df.copy()
    df["date"] = pd.PeriodIndex.from_fields(
        year=df["year"], quarter=df["quarter"]
    ).to_timestamp()
    group_col = "district_code" if "district_code" in df.columns else "neighborhood_code"
    df = df.sort_values([group_col, "date"])
    return df


def add_features(df, group_col):
    df = df.copy()

    df["contract_growth_yoy"] = df.groupby(group_col)["num_contracts"].pct_change(4, fill_method=None)
    df["rent_growth_yoy"]     = df.groupby(group_col)["avg_rent"].pct_change(4, fill_method=None)
    df["rent_m2_growth_yoy"]  = df.groupby(group_col)["avg_rent_m2"].pct_change(4, fill_method=None)
    df["contract_growth_qoq"] = df.groupby(group_col)["num_contracts"].pct_change(1, fill_method=None)

    df["rent_m2_lag1"] = df.groupby(group_col)["avg_rent_m2"].shift(1)
    df["rent_m2_lag4"] = df.groupby(group_col)["avg_rent_m2"].shift(4)

    df["post_regulation"] = (df["date"] >= "2020-10-01").astype(int)
    df["covid_dummy"]     = ((df["date"] >= "2020-03-01") & (df["date"] <= "2021-06-01")).astype(int)

    return df


districts    = add_features(create_time_index(districts_raw),    "district_code")
neighborhood = add_features(create_time_index(neighborhood_raw), "neighborhood_code")

print("districts shape:", districts.shape)
print("neighborhood shape:", neighborhood.shape)

districts shape: (1040, 17)
neighborhood shape: (3504, 17)


## IPC — Índice de Precios al Consumidor (INE)

Serie mensual → trimestral. Fuente: `ipc_ine.csv` (separador `;`, encoding latin-1).  
El IPC comienza en 2002; las observaciones anteriores quedarán como `NaN` en las variables reales.

In [10]:
ipc = pd.read_csv(IPC_FILE, sep=";", encoding="latin-1", decimal=",")

ipc = ipc.rename(columns={
    "Comunidades y Ciudades Autónomas": "region",
    "Grupos COICOP 2011": "group",
    "Tipo de dato": "data_type",
    "Periodo": "period",
    "Total": "ipc_index"
})

ipc["year"]  = ipc["period"].str[:4].astype(int)
ipc["month"] = ipc["period"].str[-2:].astype(int)
ipc["date"]  = pd.to_datetime(ipc["year"].astype(str) + "-" + ipc["month"].astype(str) + "-01")
ipc = ipc.sort_values("date").reset_index(drop=True)
ipc = ipc[["date", "ipc_index", "year", "month"]]

# Trimestral (media de los 3 meses de cada trimestre)
ipc_valid = ipc.dropna(subset=["ipc_index"]).copy()
ipc_valid["quarter"] = ipc_valid["date"].dt.quarter

ipc_quarterly = (
    ipc_valid
    .groupby(["year", "quarter"], as_index=False)
    .agg(ipc_index_q=("ipc_index", "mean"))
)
ipc_quarterly["date"] = pd.to_datetime(
    ipc_quarterly["year"].astype(str) + "-" +
    ((ipc_quarterly["quarter"] - 1) * 3 + 1).astype(str) + "-01"
)
ipc_quarterly = ipc_quarterly[["date", "year", "quarter", "ipc_index_q"]]

print(ipc_quarterly.head())
print(ipc_quarterly.tail())
print("Shape:", ipc_quarterly.shape)

        date  year  quarter  ipc_index_q
0 2002-01-01  2002        1    56.727667
1 2002-04-01  2002        2    57.955667
2 2002-07-01  2002        3    57.872667
3 2002-10-01  2002        4    58.812000
4 2003-01-01  2003        1    59.107667
         date  year  quarter  ipc_index_q
92 2025-01-01  2025        1    98.987000
93 2025-04-01  2025        2    99.982667
94 2025-07-01  2025        3   100.304333
95 2025-10-01  2025        4   100.726000
96 2026-01-01  2026        1   100.863500
Shape: (97, 4)


## Euríbor 12M (BDE)

Serie mensual → trimestral. Fuente: `SeriesBIE[21].csv` (BDE, skiprows=8).  
**MERCADO MONETARIO. TIPOS DE INTERÉS. EURIBOR. A DOCE MESES** — desde ene-99.

In [11]:
euribor = pd.read_csv(
    EURIBOR_FILE, sep=",", encoding="latin-1",
    skiprows=8, header=None, names=["period_str", "euribor_12m"]
)

month_map = {
    "ene": "01", "feb": "02", "mar": "03", "abr": "04",
    "may": "05", "jun": "06", "jul": "07", "ago": "08",
    "sep": "09", "oct": "10", "nov": "11", "dic": "12"
}

euribor["period_str"] = euribor["period_str"].astype(str).str.strip().str.lower()
euribor = euribor[
    euribor["period_str"].str.match(
        r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\d{2}$", na=False
    )
].copy()

euribor["euribor_12m"] = pd.to_numeric(euribor["euribor_12m"], errors="coerce")
euribor["month"]       = euribor["period_str"].str[:3].map(month_map)
euribor["year"]        = 2000 + euribor["period_str"].str[3:5].astype(int)
euribor["date"]        = pd.to_datetime(
    euribor["year"].astype(str) + "-" + euribor["month"] + "-01"
)
euribor = (
    euribor[["date", "euribor_12m"]]
    .dropna(subset=["euribor_12m"])
    .loc[lambda d: d["date"] < "2026-01-01"]
    .sort_values("date")
    .reset_index(drop=True)
)

# Trimestral
euribor["year"]    = euribor["date"].dt.year
euribor["quarter"] = euribor["date"].dt.quarter

euribor_quarterly = (
    euribor
    .groupby(["year", "quarter"], as_index=False)
    .agg(euribor_12m_q=("euribor_12m", "mean"))
)
euribor_quarterly["date"] = pd.to_datetime(
    euribor_quarterly["year"].astype(str) + "-" +
    ((euribor_quarterly["quarter"] - 1) * 3 + 1).astype(str) + "-01"
)
euribor_quarterly = euribor_quarterly[["date", "year", "quarter", "euribor_12m_q"]]

print(euribor_quarterly.head())
print(euribor_quarterly.tail())
print("Shape:", euribor_quarterly.shape)

        date  year  quarter  euribor_12m_q
0 2000-01-01  2000        1       4.109000
1 2000-04-01  2000        2       4.726333
2 2000-07-01  2000        3       5.190667
3 2000-10-01  2000        4       5.097333
4 2001-01-01  2001        1       4.545333
          date  year  quarter  euribor_12m_q
99  2024-10-01  2024        4       2.544333
100 2025-01-01  2025        1       2.443333
101 2025-04-01  2025        2       2.101667
102 2025-07-01  2025        3       2.121667
103 2025-10-01  2025        4       2.223667
Shape: (104, 4)


## Merge IPC + precios reales + YoY real

In [12]:
def add_real_rent_variables(df, rent_col="avg_rent", rent_m2_col="avg_rent_m2",
                             ipc_col="ipc_index_q"):
    out = df.copy()
    out["avg_rent_real_2025base"]    = out[rent_col]    / (out[ipc_col] / 100)
    out["avg_rent_m2_real_2025base"] = out[rent_m2_col] / (out[ipc_col] / 100)
    return out


def add_real_yoy_growth(df, group_col,
                        rent_real_col="avg_rent_real_2025base",
                        rent_m2_real_col="avg_rent_m2_real_2025base"):
    out = df.copy().sort_values([group_col, "date"])
    out["rent_real_growth_yoy"] = (
        out.groupby(group_col)[rent_real_col]
        .pct_change(periods=4, fill_method=None)
    )
    out["rent_m2_real_growth_yoy"] = (
        out.groupby(group_col)[rent_m2_real_col]
        .pct_change(periods=4, fill_method=None)
    )
    return out


# Merge IPC
districts_ipc    = districts.merge(ipc_quarterly[["date", "ipc_index_q"]], on="date", how="left")
neighborhood_ipc = neighborhood.merge(ipc_quarterly[["date", "ipc_index_q"]], on="date", how="left")

print("NaN IPC districts:",    districts_ipc["ipc_index_q"].isna().sum())
print("NaN IPC neighborhood:", neighborhood_ipc["ipc_index_q"].isna().sum())

# Precios reales
districts_ipc    = add_real_rent_variables(districts_ipc)
neighborhood_ipc = add_real_rent_variables(neighborhood_ipc)

# YoY reales
districts_ipc    = add_real_yoy_growth(districts_ipc,    "district_code")
neighborhood_ipc = add_real_yoy_growth(neighborhood_ipc, "neighborhood_code")

print("\nDistintas variables reales calculadas.")
print(districts_ipc[["date", "avg_rent", "avg_rent_real_2025base",
                       "avg_rent_m2", "avg_rent_m2_real_2025base"]].head())

NaN IPC districts: 80
NaN IPC neighborhood: 0

Distintas variables reales calculadas.
        date  avg_rent  avg_rent_real_2025base  avg_rent_m2  \
0 2000-01-01  286.9246                     NaN     4.998756   
1 2000-04-01  301.3140                     NaN     5.401631   
2 2000-07-01  338.7222                     NaN     5.958517   
3 2000-10-01  370.0641                     NaN     5.758672   
4 2001-01-01  345.2979                     NaN     6.459721   

   avg_rent_m2_real_2025base  
0                        NaN  
1                        NaN  
2                        NaN  
3                        NaN  
4                        NaN  


## Merge Euríbor

In [13]:
districts_ipc_euribor    = districts_ipc.merge(
    euribor_quarterly[["date", "euribor_12m_q"]], on="date", how="left"
)
neighborhood_ipc_euribor = neighborhood_ipc.merge(
    euribor_quarterly[["date", "euribor_12m_q"]], on="date", how="left"
)

print("NaN Euribor districts:",    districts_ipc_euribor["euribor_12m_q"].isna().sum())
print("NaN Euribor neighborhood:", neighborhood_ipc_euribor["euribor_12m_q"].isna().sum())

# Verificar duplicados
print("Duplicados districts:",    districts_ipc_euribor.duplicated(subset=["district_code", "date"]).sum())
print("Duplicados neighborhood:", neighborhood_ipc_euribor.duplicated(subset=["neighborhood_code", "date"]).sum())

NaN Euribor districts: 0
NaN Euribor neighborhood: 0
Duplicados districts: 0
Duplicados neighborhood: 0


## IST — Índice Socioeconómico Territorial (Idescat)

Variable estructural por barrio. Disponible desde 2015; los años anteriores y los más recientes quedarán como `NaN`.

In [14]:
ist_neighborhood = pd.read_csv(IST_FILE, sep=";", encoding="utf-8-sig", decimal=",")

ist_neighborhood = ist_neighborhood.rename(columns={
    "año": "year",
    "municipio": "municipality",
    "barrios de Barcelona": "neighborhood_name",
    "concepto": "concept",
    "estado": "status",
    "valor": "ist_index"
})

ist_neighborhood_clean = ist_neighborhood[["year", "neighborhood_name", "ist_index"]].copy()


def normalize_text(text):
    if pd.isna(text):
        return text
    text = str(text).strip().lower()
    text = "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )
    text = text.replace("\u2019", "'").replace("`", "'")
    return " ".join(text.split())


ist_neighborhood_clean["neighborhood_name_norm"] = (
    ist_neighborhood_clean["neighborhood_name"].apply(normalize_text)
)
ist_neighborhood_clean = ist_neighborhood_clean[
    ist_neighborhood_clean["neighborhood_name_norm"] != "total"
].copy()

# Mapeo manual de nombres divergentes entre IST y panel
manual_name_map = {
    "el poble sec":                  "el poble sec - aei parc montjuic",
    "la marina del prat vermell":    "la marina del prat vermell - aei zona franca",
    "sant gervasi-galvany":          "sant gervasi - galvany",
    "sant gervasi-la bonanova":      "sant gervasi - la bonanova",
    "sants-badal":                   "sants - badal",
}
ist_neighborhood_clean["neighborhood_norm_matched"] = (
    ist_neighborhood_clean["neighborhood_name_norm"].replace(manual_name_map)
)

# Normalizar nombres del panel
neighborhood_ipc_euribor["neighborhood_norm"] = (
    neighborhood_ipc_euribor["neighborhood"].apply(normalize_text)
)

neighborhood_ipc_euribor_ist = neighborhood_ipc_euribor.merge(
    ist_neighborhood_clean[["year", "neighborhood_norm_matched", "ist_index"]],
    left_on=["year", "neighborhood_norm"],
    right_on=["year", "neighborhood_norm_matched"],
    how="left"
).drop(columns=["neighborhood_norm_matched"])

print("NaN IST:", neighborhood_ipc_euribor_ist["ist_index"].isna().sum())
print("Barrios únicos:", neighborhood_ipc_euribor_ist["neighborhood"].nunique())

NaN IST: 876
Barrios únicos: 73


## Población y densidad (Open Data BCN)

Datos anuales de padrón municipal por sección censal → agregados a barrio y distrito.  
Superficie de 2021 tratada como fija para calcular densidad (hab/ha).

In [15]:
# Recursos de población disponibles 
resp = requests.get(
    "https://opendata-ajuntament.barcelona.cat/data/api/3/action/package_show",
    params={"id": "pad_mdbas"},
    timeout=30
)
resp.raise_for_status()
data = resp.json()

resources_df = pd.DataFrame([
    {"resource_id": r.get("id"), "name": r.get("name"),
     "format": r.get("format"), "url": r.get("url")}
    for r in data["result"]["resources"]
])

resources_csv = resources_df[resources_df["format"].str.upper().eq("CSV")].copy()
resources_csv["year"] = (
    resources_csv["name"].str.extract(r"(\d{4})", expand=False).astype("Int64")
)

pop_resources = (
    resources_csv.loc[resources_csv["year"].between(2000, 2025)]
    .sort_values("year")
    .reset_index(drop=True)
)

print(pop_resources[["year", "name", "url"]].to_string())

    year                name                                                                                                                                                 url
0   2000  2000_pad_mdbas.csv  https://opendata-ajuntament.barcelona.cat/data/dataset/2f6e0561-30f4-44a0-8446-e27442d4754c/resource/3c21fd04-5f98-41ce-b1cb-56e2b7cf0c84/download
1   2001  2001_pad_mdbas.csv  https://opendata-ajuntament.barcelona.cat/data/dataset/2f6e0561-30f4-44a0-8446-e27442d4754c/resource/8a602477-d724-4a82-9b87-3e42dba55040/download
2   2002  2002_pad_mdbas.csv  https://opendata-ajuntament.barcelona.cat/data/dataset/2f6e0561-30f4-44a0-8446-e27442d4754c/resource/411c5de4-6b6d-4228-b159-4c1d96b8e2cb/download
3   2003  2003_pad_mdbas.csv  https://opendata-ajuntament.barcelona.cat/data/dataset/2f6e0561-30f4-44a0-8446-e27442d4754c/resource/816ec8d3-43b4-4da0-864f-384ef6e6f6a7/download
4   2004  2004_pad_mdbas.csv  https://opendata-ajuntament.barcelona.cat/data/dataset/2f6e0561-30f4-44a0-8446-e27442

In [16]:
def load_population_resource(url):
    df = pd.read_csv(url, sep=",", encoding="utf-8-sig")
    df = df.rename(columns={
        "Data_Referencia": "date",
        "Codi_Districte": "district_code",
        "Nom_Districte": "district",
        "Codi_Barri": "neighborhood_code",
        "Nom_Barri": "neighborhood",
        "AEB": "aeb",
        "Seccio_Censal": "census_section",
        "Valor": "population"
    })
    df["date"]       = pd.to_datetime(df["date"])
    df["year"]       = df["date"].dt.year
    df["population"] = pd.to_numeric(df["population"], errors="coerce")
    return df


population_all = pd.concat(
    [load_population_resource(url) for url in pop_resources["url"]],
    ignore_index=True
)

print(f"Población cargada: {population_all.shape}  "
      f"Años: {population_all['year'].min()}–{population_all['year'].max()}")

# Agregación a barrio y distrito
population_neighborhood = (
    population_all
    .groupby(["year", "district_code", "district", "neighborhood_code", "neighborhood"],
             as_index=False)
    .agg(population_total=("population", "sum"))
)

population_district = (
    population_all
    .groupby(["year", "district_code", "district"], as_index=False)
    .agg(population_total=("population", "sum"))
)

print("Barrios únicos:", population_neighborhood["neighborhood_code"].nunique())
print("Distritos únicos:", population_district["district_code"].nunique())

Población cargada: (27766, 9)  Años: 2000–2025
Barrios únicos: 73
Distritos únicos: 10


In [17]:
#  Superficie (dataset est-superficie, año 2021 como fija) 
resp_surf = requests.get(
    "https://opendata-ajuntament.barcelona.cat/data/api/3/action/package_show",
    params={"id": "est-superficie"},
    timeout=30
)
resp_surf.raise_for_status()
surf_resources = pd.DataFrame([
    {"name": r.get("name"), "format": r.get("format"), "url": r.get("url")}
    for r in resp_surf.json()["result"]["resources"]
])

url_surf_2021 = surf_resources.loc[
    surf_resources["name"].str.contains("2021", case=False, na=False), "url"
].iloc[0]

surface_raw = pd.read_csv(url_surf_2021, sep=",", encoding="utf-8-sig")

surface_neighborhood = surface_raw.rename(columns={
    "Any": "year", "Codi_Districte": "district_code", "Nom_Districte": "district",
    "Codi_Barri": "neighborhood_code", "Nom_Barri": "neighborhood",
    "Superfície (ha)": "surface_ha"
})[["district_code", "district", "neighborhood_code", "neighborhood", "surface_ha"]].copy()

surface_district = (
    surface_neighborhood
    .groupby(["district_code", "district"], as_index=False)
    .agg(surface_ha=("surface_ha", "sum"))
)

#  Densidad 
population_neighborhood_density = population_neighborhood.merge(
    surface_neighborhood[["neighborhood_code", "surface_ha"]], on="neighborhood_code", how="left"
)
population_neighborhood_density["population_density_ha"] = (
    population_neighborhood_density["population_total"] /
    population_neighborhood_density["surface_ha"]
)

population_district_density = population_district.merge(
    surface_district[["district_code", "surface_ha"]], on="district_code", how="left"
)
population_district_density["population_density_ha"] = (
    population_district_density["population_total"] /
    population_district_density["surface_ha"]
)

print("NaN density neighborhood:", population_neighborhood_density["population_density_ha"].isna().sum())
print("NaN density district:",     population_district_density["population_density_ha"].isna().sum())

NaN density neighborhood: 0
NaN density district: 0


## Merge final + guardar parquets de salida

In [18]:
districts_final = districts_ipc_euribor.merge(
    population_district_density[["year", "district_code", "population_total", "population_density_ha"]],
    on=["year", "district_code"],
    how="left"
)

neighborhood_final = neighborhood_ipc_euribor_ist.drop(
    columns=["neighborhood_norm"], errors="ignore"
).merge(
    population_neighborhood_density[["year", "neighborhood_code", "population_total", "population_density_ha"]],
    on=["year", "neighborhood_code"],
    how="left"
)

print("districts_final:", districts_final.shape)
print("NaN density districts:", districts_final["population_density_ha"].isna().sum())
print()
print("neighborhood_final:", neighborhood_final.shape)
print("NaN density neighborhood:", neighborhood_final["population_density_ha"].isna().sum())

# Guardar parquets finales
# districts_final.to_parquet("districts_final.parquet", index=False)
# neighborhood_final.to_parquet("neighborhood_final.parquet", index=False)

print("\nMerge TRANSFORM completado (guardado en disco desactivado).")

districts_final: (1040, 25)
NaN density districts: 0

neighborhood_final: (3504, 26)
NaN density neighborhood: 0

Merge TRANSFORM completado (guardado en disco desactivado).


# LOAD
Limpieza final, tipado, etiquetas temporales y construcción de datasets listos para el EDA y modelado.

Los DataFrames `districts_final` y `neighborhood_final` vienen directamente de la etapa TRANSFORM (en memoria).

## Limpieza — Distritos

In [19]:
df_distritos = districts_final.copy()

# Filas con nulos en variables originales de Incasol
filtro_null_distrito = df_distritos[["num_contracts", "avg_rent", "avg_rent_m2", "avg_surface"]].isnull()

print("Filas con valores nulos en variables originales de distritos:")
print(df_distritos.loc[filtro_null_distrito.any(axis=1), ["district", "year", "quarter"]])

df_distritos_clean = df_distritos[~filtro_null_distrito.any(axis=1)].copy()
print(f"\nDistrictos clean: {df_distritos_clean.shape}")

Filas con valores nulos en variables originales de distritos:
                 district  year  quarter
103          Ciutat Vella  2025        4
207              Eixample  2025        4
311        Sants-Montjuïc  2025        4
415             Les Corts  2025        4
519   Sarrià-Sant Gervasi  2025        4
623                Gràcia  2025        4
727        Horta-Guinardó  2025        4
831            Nou Barris  2025        4
935           Sant Andreu  2025        4
1039           Sant Martí  2025        4

Districtos clean: (1030, 25)


## Limpieza — Barrios

In [20]:
df_barrios = neighborhood_final.copy()

# Excluir 2025Q4 (dato aún incompleto)
df_barrios_clean = df_barrios[
    ~((df_barrios["year"] == 2025) & (df_barrios["quarter"] == 4))
].copy()

# Barrios con series demasiado escasas para análisis robusto
barrios_problematicos = [
    "Baró de Viver", "Can Peguera", "Canyelles",
    "Torre Baró", "Vallbona", "la Clota",
    "la Marina del Prat Vermell - AEI Zona Franca"
]

df_barrios_clean_final = df_barrios_clean.loc[
    ~df_barrios_clean["neighborhood"].isin(barrios_problematicos)
].copy()

print(f"Barrios clean final: {df_barrios_clean_final.shape}")
print(f"Barrios únicos: {df_barrios_clean_final['neighborhood'].nunique()}")
print(f"Valores nulos:\n{df_barrios_clean_final.isnull().sum()[lambda s: s > 0]}")

Barrios clean final: (3102, 26)
Barrios únicos: 66
Valores nulos:
contract_growth_yoy        264
rent_growth_yoy            264
rent_m2_growth_yoy         264
contract_growth_qoq         66
rent_m2_lag1                66
rent_m2_lag4               264
rent_real_growth_yoy       264
rent_m2_real_growth_yoy    264
ist_index                  726
dtype: int64


## Tipado y etiquetas temporales — Distritos

In [21]:
df_distritos_eda = df_distritos_clean.copy()

df_distritos_eda["district"]       = df_distritos_eda["district"].astype("category")
df_distritos_eda["quarter"]        = df_distritos_eda["quarter"].astype("category")
df_distritos_eda["post_regulation"] = df_distritos_eda["post_regulation"].astype(bool)
df_distritos_eda["covid_dummy"]     = df_distritos_eda["covid_dummy"].astype(bool)

reg_start = pd.Timestamp("2020-10-01")

df_distritos_eda["period_prepost"] = pd.Categorical(
    np.where(df_distritos_eda["date"] < reg_start, "Pre-regulation", "Post-regulation"),
    categories=["Pre-regulation", "Post-regulation"],
    ordered=True
)

quarter_num = df_distritos_eda["quarter"].astype(int)

conditions = [
    (df_distritos_eda["year"] <= 2019),
    (df_distritos_eda["year"] == 2020) & (quarter_num.isin([1, 2, 3])),
    ((df_distritos_eda["year"] == 2020) & (quarter_num == 4)) | (df_distritos_eda["year"] == 2021),
    (df_distritos_eda["year"] >= 2022),
]
choices = [
    "Pre-COVID / pre-regulation",
    "COVID shock",
    "Regulation + COVID / transition",
    "Post-overlap / new regime",
]
df_distritos_eda["period_4levels"] = pd.Categorical(
    np.select(conditions, choices, default="Unknown"),
    categories=choices,
    ordered=True
)

print("df_distritos_eda listo:", df_distritos_eda.shape)
df_distritos_eda.info()

df_distritos_eda listo: (1030, 27)
<class 'pandas.core.frame.DataFrame'>
Index: 1030 entries, 0 to 1038
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   district_code              1030 non-null   int64         
 1   district                   1030 non-null   category      
 2   year                       1030 non-null   int64         
 3   quarter                    1030 non-null   category      
 4   num_contracts              1030 non-null   Int64         
 5   avg_rent                   1030 non-null   float64       
 6   avg_rent_m2                1030 non-null   float64       
 7   avg_surface                1030 non-null   float64       
 8   date                       1030 non-null   datetime64[ns]
 9   contract_growth_yoy        990 non-null    Float64       
 10  rent_growth_yoy            990 non-null    float64       
 11  rent_m2_growth_yoy         990 non-null

## Tipado y etiquetas temporales — Barrios

In [22]:
df_barrios_eda = df_barrios_clean_final.copy()

for col, dtype in [("district", "category"), ("neighborhood", "category"), ("quarter", "category")]:
    if col in df_barrios_eda.columns:
        df_barrios_eda[col] = df_barrios_eda[col].astype(dtype)

for col in ["post_regulation", "covid_dummy"]:
    if col in df_barrios_eda.columns:
        df_barrios_eda[col] = df_barrios_eda[col].astype(bool)

quarter_num_b = df_barrios_eda["quarter"].astype(int)

df_barrios_eda["period_prepost"] = pd.Categorical(
    np.where(df_barrios_eda["date"] < reg_start, "Pre-regulation", "Post-regulation"),
    categories=["Pre-regulation", "Post-regulation"],
    ordered=True
)

conditions_b = [
    (df_barrios_eda["year"] <= 2019),
    (df_barrios_eda["year"] == 2020) & (quarter_num_b.isin([1, 2, 3])),
    ((df_barrios_eda["year"] == 2020) & (quarter_num_b == 4)) | (df_barrios_eda["year"] == 2021),
    (df_barrios_eda["year"] >= 2022),
]
df_barrios_eda["period_4levels"] = pd.Categorical(
    np.select(conditions_b, choices, default="Unknown"),
    categories=choices,
    ordered=True
)

print("df_barrios_eda listo:", df_barrios_eda.shape)
df_barrios_eda.info()

df_barrios_eda listo: (3102, 28)
<class 'pandas.core.frame.DataFrame'>
Index: 3102 entries, 0 to 3502
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   neighborhood_code          3102 non-null   int64         
 1   neighborhood               3102 non-null   category      
 2   year                       3102 non-null   int64         
 3   quarter                    3102 non-null   category      
 4   num_contracts              3102 non-null   Int64         
 5   avg_rent                   3102 non-null   float64       
 6   avg_rent_m2                3102 non-null   float64       
 7   avg_surface                3102 non-null   float64       
 8   date                       3102 non-null   datetime64[ns]
 9   contract_growth_yoy        2838 non-null   Float64       
 10  rent_growth_yoy            2838 non-null   float64       
 11  rent_m2_growth_yoy         2838 non-null 

## Resumen final del pipeline

| Output | Descripción |
|---|---|
| `df_distritos_eda` | Panel distritos listo para EDA/modelado |
| `df_barrios_eda` | Panel barrios listo para EDA/modelado |

In [23]:
print("=" * 60)
print("ETL COMPLETO — OUTPUTS FINALES")
print("=" * 60)

print(f"\ndf_distritos_eda : {df_distritos_eda.shape}")
print(f"  Distritos únicos : {df_distritos_eda['district'].nunique()}")
print(f"  Periodo          : {df_distritos_eda['year'].min()}–{df_distritos_eda['year'].max()}")
print(f"  Columnas         : {df_distritos_eda.columns.tolist()}")

print(f"\ndf_barrios_eda   : {df_barrios_eda.shape}")
print(f"  Barrios únicos   : {df_barrios_eda['neighborhood'].nunique()}")
print(f"  Periodo          : {df_barrios_eda['year'].min()}–{df_barrios_eda['year'].max()}")
print(f"  Columnas         : {df_barrios_eda.columns.tolist()}")


# Guardar outputs finales en disco. Descomentar para guardar.
# df_distritos_eda.to_parquet("df_distritos_eda.parquet", index=False)
# df_barrios_eda.to_parquet("df_barrios_eda.parquet", index=False)

#print("\nOutputs guardados: df_distritos_eda.parquet, df_barrios_eda.parquet")

ETL COMPLETO — OUTPUTS FINALES

df_distritos_eda : (1030, 27)
  Distritos únicos : 10
  Periodo          : 2000–2025
  Columnas         : ['district_code', 'district', 'year', 'quarter', 'num_contracts', 'avg_rent', 'avg_rent_m2', 'avg_surface', 'date', 'contract_growth_yoy', 'rent_growth_yoy', 'rent_m2_growth_yoy', 'contract_growth_qoq', 'rent_m2_lag1', 'rent_m2_lag4', 'post_regulation', 'covid_dummy', 'ipc_index_q', 'avg_rent_real_2025base', 'avg_rent_m2_real_2025base', 'rent_real_growth_yoy', 'rent_m2_real_growth_yoy', 'euribor_12m_q', 'population_total', 'population_density_ha', 'period_prepost', 'period_4levels']

df_barrios_eda   : (3102, 28)
  Barrios únicos   : 66
  Periodo          : 2014–2025
  Columnas         : ['neighborhood_code', 'neighborhood', 'year', 'quarter', 'num_contracts', 'avg_rent', 'avg_rent_m2', 'avg_surface', 'date', 'contract_growth_yoy', 'rent_growth_yoy', 'rent_m2_growth_yoy', 'contract_growth_qoq', 'rent_m2_lag1', 'rent_m2_lag4', 'post_regulation', 'covi